# Measles infection in the 1970 British Cohort Study

## 1. Load packages, load data, merge dataframes

In [ ]:
## Load packages, install if needed (if using Anaconda, need to install them directly thru conda terminal)
library(tidyverse)
library(ggplot2)
library(broom)
library(epitools)
library(ggeffects)
library(gtsummary)
library(flextable)
library(patchwork)
library(gt)
library(svglite)
library(purrr)
library(forcats)
library(cardx)
library(survey)

#ipw
library(cobalt)
library(sandwich)
library(lmtest)


In [ ]:
## Load data

## Link to data: https://unioxfordnexus-my.sharepoint.com/:f:/r/personal/wolf7427_ox_ac_uk/Documents/oxford_dphil/projects/measles/manuscript/input_tables?csf=1&web=1&e=UXloSR
base_path = "manuscript/input_tables"

files = c(
  "bcs1derived.tab", "bcs7072a.tab", "bcs7072b.tab", "bcs2derived.tab",
  "f699a.tab", "f699b.tab", "bcs3derived.tab", "sn3723.tab",
  "bcs4derived.tab", "bcs7016x.tab", "bcs_age46_main.tab",
  "bcs5derived.tab", "bcs96x.tab", "bcs6derived.tab", "bcs2000.tab",
  "bcs_2004_followup.tab", "bcs8derived.tab", "bcs_2008_followup.tab",
  "bcs70_2012_derived.tab", "bcs70_2012_flatfile.tab", "bcs1986_reading_matrices.tab",
  "bcs70_16_year_arithmetic_data.tab","bcs_2004_adult_assessment_basic_skills.tab"
)

# Read and assign each file as a data frame with df_ prefix
for (file in files) {
    name = paste0("df_", tools::file_path_sans_ext(file))  # removes .tab extension
    path = file.path(base_path, file)
    assign(name, read_tsv(path))
}

In [ ]:
## Merge dataframes into appropriate sweep

##CHECK THE DATAFRAMES TO MAKE SURE YOU ACTUALLY NEED THEM. REMOVE UNNECCESSARY DATAFRAMES 

#Birth sweep
df_birth = df_bcs1derived %>% inner_join(df_bcs7072a, by = c("BCSID" = "bcsid"))
#nrow = 17194

#Age 5 sweep
df_age5 = df_bcs2derived %>% 
    inner_join(df_f699a, by = c("BCSID" = "bcsid")) %>%
    inner_join(df_f699b, by = c("BCSID" = "bcsid"))
#nrow = 13133

#Age 10 sweep
df_age10 = df_bcs3derived %>% inner_join(df_sn3723, by = "bcsid")
#nrow = 14868

#Age 16 sweep
df_age16 = df_bcs4derived %>% inner_join(df_bcs7016x, by = c("BCSID" = "bcsid"))
#nrow = 11614

#Age 16 cognitive
df_16cog = df_bcs1986_reading_matrices %>% inner_join(df_bcs70_16_year_arithmetic_data, by = c("BCSID" = "bcsid"))

#Age 26 sweep
df_age26 = df_bcs5derived %>% inner_join(df_bcs96x, by = c("BCSID" = "bcsid"))
#nrow = 9002

#Age 29 sweep
df_age29 = df_bcs6derived %>% inner_join(df_bcs2000, by = c("BCSID" = "bcsid"))
#nrow = 11076

#Age 34 sweep
df_age34 = df_bcs_2004_followup
#nrow = 9663

df_age34cog = df_bcs_2004_adult_assessment_basic_skills

#Age 38 sweep
df_age38 = df_bcs8derived %>% inner_join(df_bcs_2008_followup, by = c("BCSID" = "bcsid"))
#nrow = 8873

#Age 42 sweep
df_age42 = df_bcs70_2012_derived %>% inner_join(df_bcs70_2012_flatfile, by = "BCSID")
#nrow = 9002

#Age 46 sweep
df_age46 = df_bcs_age46_main
#nrow = 9840


In [ ]:
##Merge all dataframes
df_merged = df_birth %>%
  full_join(df_age5, by = "BCSID") %>%
  full_join(df_age10, by = c("BCSID" = "bcsid")) %>%
  full_join(df_age16, by = "BCSID") %>%
  full_join(df_16cog, by = "BCSID") %>%
  full_join(df_age26, by = "BCSID") %>%
  full_join(df_age29, by = "BCSID") %>%
  full_join(df_age34, by = c("BCSID" = "bcsid")) %>%
  full_join(df_age34cog, by = c("BCSID" = "bcsid")) %>%
  full_join(df_age38, by = "BCSID") %>%
  full_join(df_age42, by = "BCSID") %>%
  full_join(df_age46, by = "BCSID")

In [ ]:
columns <- c(

  # IDENTIFIERS
  "BCSID",
    
  # MEASLES INFECTION
  "b12.1", "ob11.2",
    
  # CHARACTERISTICS
  "sex10.x",       # Sex from age 10
  "BD1REGN",       # Location/region
  "BD2REGN",       # Location age 5
  "bd3regn",       # Location/region age 10
  "e245",          # Ethnic group
  "a12.1",         # Ethnic group age 10
  "BD1PSOC",       # Social index
  "BD2SOC",        # Social index age 5
  "bd3psoc",       # Social index age 10

  "a0043b",        # Smoking during pregnancy
  "e10.1",

  "B10SMOKIG", "B9SMOKIG", "bd8smoke", "bd7smoke", "smoking", "b960632",  # Adult smoking
    
  "bd3ben", "bd3inc",      # Income age 10

  "c1.1", "c1.2", "c1.3", "c1.4", "c1.5", "c1.6",     # parental Education
  "c1.7", "c1.8", "c1.9", "c1.10", "c1.11", "c1.12", 
  "c1.13", "c1.14", "c1.15", "c1.16", "c1.17", "c1.18",
  "c1.19", "c1.20", "c1.21", "c1.22", "c2.1",
    
    
  "a4a.1", "a4a.2", "a4a.3", "a4a.4", "a4a.5",         # Older and younger siblings, maternal age
  "a4a.6", "a4a.8", "a4a.9", "a4a.10", "a4a.12",        
  "a4a.13", "a4a.14", "a4a.16", "a4a.17", "a4a.18",
  "a4a.20", "a4a.21", "a4a.22", "a4a.24", "a4a.25",
  "a4a.26", "a4a.28", "a4a.29", "a4a.30", "a4a.32",
  "a4a.33", "a4a.34", "a4a.36", "a4a.37", "a4a.38",
  "a4a.40", "a4a.41", "a4a.42",

 "a4b.1","a4b.4","a4b.6","a4b.9","a4b.11","a4b.14","a4b.16","a4b.19",
    
  "BD1MAGE", "e008",      # Maternal age

  #### DIABETES
  "b7diabag", "B9AGEINS","B10AGEINS", "B9PAGENS",
  "dl112m", "B9INSULN", "B10INSULN", "B10DIABD", "B9PINSLN",
  "b960455", "b960521", "diab", "dl1age", 
  "bd7hpb04", "b7diabag", "b8khpb03", "B9KHPB04", "B9PKHP04", "B10KHPB03",
  "B10DIABD",  # Diabetes

  ### HYPERTENSION
  "downhibp", "bp1age", "bd7hpb11", "b8khpb09", "B9KHPB09", "B9PKHP09",
  "B10KHPB10", 


  # BIOMARKERS
  "B10CHOL", "B10HDL", "B10HBA1C", "B10IGF1", "B10RTIN",
  "B10TRIG", "B10RBC",
  "B10HSCRP", "B10USCMG", "B10USCMM", "B10CMVAVC",


  "B10HEIGHTCM", "B10MWEIGHT",                          # BMI (age 46)
  "B10WAISTAV", "B10HIPAV",                             # Waist / hip circumference
  "B10BPSYSR1", "B10BPSYSR2", "B10BPSYSR3",             # Systolic blood pressure
  "B10BPDIAR1", "B10BPDIAR2", "B10BPDIAR3",             # Diastolic blood pressure


  # COGNITIVE
  "bd3rread", "bd3read", "bd3rdage", "bd3maths",   # Age 10 cognitive
    
  "mathscore", "BD4RDAGE", "SCR_M", "SCRTOTAL",    # Age 16 cognitive

  "numall", "litall37",   #age 34 cog
  "B9VSCORE",            # age 42 cog
    
  "B10CFLISN", "B10CFLISD", "B10CFANI", "B10CFCOR", # Age 46 cognitive
  "B10CFMIS", "B10CFTOT", "B10CFRC",

  # ADULT EDUC ATTAINMENT
  "B9AGELE", "actagel", "b7actagl", "b960132","b960129",

  # breast feeding
  "e020"

)


In [ ]:
df_selected = df_merged %>% select(all_of(columns))

## 2. Pre-processing

In [ ]:
df_process = df_selected

#### Exposure

In [ ]:
# ------------------------------------------------------------------------------
# MEASLES INFECTION AND AGE OF ONSET
#
# This section combines information from multiple sweeps to determine whether
# the child ever had measles and at what age.
#
# Variables used:
# - b12.1: Age 10 report — has child *ever* had measles?
#   - 1 = Yes → "Yes"
#   - 2 = No  → "No"
#   - Other / missing → NA
#
# - ob11.2: Age 16 report — has teen had measles *since age 10*?
#   - If this is 1 (Yes), overwrite prior response and set measlesinfection = "Yes"
#   - Also set measles_age = 16 to reflect timing
#
#
# Final output variables:
# - measlesinfection: Factor ("No", "Yes") indicating whether measles was ever reported
# ------------------------------------------------------------------------------

## Infection (ever had measles?)
df_process$measlesinfection <- ifelse(df_process$b12.1 == 1, "Yes",
                                 ifelse(df_process$b12.1 == 2, "No", NA_character_))

# Age 16 sweep: teen has had measles since age 10 → would normally overwrite
df_process$measlesinfection[df_process$ob11.2 == 1] <- "Yes"


####### THIS IS SENSITIVITY #######
# Now identify those where measles was first reported at 16 (No → Yes or NA → Yes)
first_report_at_16 <- (df_process$ob11.2 == 1) &
                      (is.na(df_process$b12.1) | df_process$b12.1 == 2)

# Set those cases to NA instead of "Yes"
df_process$measlesinfection[first_report_at_16] <- NA_character_

##############

# Make into factor
df_process$measlesinfection <- factor(df_process$measlesinfection, levels = c("No", "Yes"))


In [ ]:
summary(df_process$measlesinfection)

In [ ]:
## Check how many new cases of measles from 10 to 16
## Recreate BEFORE (age 10) and AFTER (after age 16 update)
before <- ifelse(df_process$b12.1 == 1, "Yes",
          ifelse(df_process$b12.1 == 2, "No", NA))

after <- before
after[df_process$ob11.2 == 1] <- "Yes"

## Transition table with NA as its own level
tab <- table(
  Before = addNA(factor(before, levels = c("Yes","No"))),
  After  = addNA(factor(after,  levels = c("Yes","No")))
)
tab

## Specific transitions:
no_to_yes <- tab["No", "Yes"]          # "No" → "Yes"
na_to_yes <- tab["<NA>", "Yes"]        # "NA" → "Yes"
no_to_yes; na_to_yes


#### Covariates

Sex, ethnicity, location, social index, maternal smoking, maternal age, and # of older children

In [ ]:
##COVARIATES

### SEX
df_process <- df_process %>%
  mutate(
    sex = factor(
      case_when(
        sex10.x == 1 ~ 0L,                         # Male → 0
        sex10.x == 2 ~ 1L,                         # Female → 1
        sex10.x %in% c(-1, -2, -3) ~ NA_integer_   # Missing/unknown → NA
      ),
      levels = c(0, 1),
      labels = c("Male", "Female")
    )
  )




### ETHNICITY FROM age 10 or 5
df_process <- df_process %>%
  mutate(
    # Recode age10 ethnicity (a12.1)
    ethnicity_10 = case_when(
      a12.1 %in% c(1, 2) ~ "European UK",           # English, Irish
      a12.1 == 3 ~ "European Other",                # Other European
      a12.1 == 4 ~ "West Indian",
      a12.1 %in% c(5, 6) ~ "Indian-Pakistani",      # Indian, Pakistani
      a12.1 == 7 ~ "Other Asian",                   # Bangladeshi
      a12.1 == 8 ~ "Other",
      a12.1 %in% c(-1, -2, -3, -4) ~ NA_character_,
      TRUE ~ NA_character_
    ),
    
    # Recode age 5 ethnicity (e245)
    ethnicity_5 = case_when(
      e245 == 1 ~ "European UK",
      e245 == 2 ~ "European Other",
      e245 == 3 ~ "West Indian",
      e245 == 4 ~ "Indian-Pakistani",
      e245 == 5 ~ "Other Asian",
      e245 == 6 ~ "African",
      e245 == 7 ~ "Other",
      e245 %in% c(-1, -2, -3, -4) ~ NA_character_,
      TRUE ~ NA_character_
    ),
    
    # Prefer age10 if available, otherwise age5
    ethnicity = coalesce(ethnicity_10, ethnicity_5),
    
    # Binary recode
    ethnicity_binary_num = case_when(
      ethnicity == "European UK" ~ 0L,
      ethnicity %in% c("European Other", "West Indian", "Indian-Pakistani", "Other Asian", "African", "Other") ~ 1L,
    ),
    
    ethnicity_binary = factor(
      ethnicity_binary_num,
      levels = c(0, 1),
      labels = c("European UK", "Other")
    )
  )


### LOCATION
df_process <- df_process %>%
  mutate(
    # --- age10 (bd3regn) ---
    location_bd3 = case_when(
      bd3regn == 1  ~ "North",
      bd3regn == 2  ~ "Yorks and Humberside",
      bd3regn == 3  ~ "East Midlands",
      bd3regn == 4  ~ "East Anglia",
      bd3regn == 5  ~ "South East",
      bd3regn == 6  ~ "South West",
      bd3regn == 7  ~ "West Midlands",
      bd3regn == 8  ~ "North West",
      bd3regn == 9  ~ "Wales",
      bd3regn == 10 ~ "Scotland",
      bd3regn %in% c(-1, -2, 11, 12) ~ NA_character_,
      TRUE ~ NA_character_
    ),

    # --- 1975 (BD2REGN) ---
    location_bd2 = case_when(
      BD2REGN == 1  ~ "North",
      BD2REGN == 2  ~ "Yorks and Humberside",
      BD2REGN == 3  ~ "East Midlands",
      BD2REGN == 4  ~ "East Anglia",
      BD2REGN == 5  ~ "South East",
      BD2REGN == 6  ~ "South West",
      BD2REGN == 7  ~ "West Midlands",
      BD2REGN == 8  ~ "North West",
      BD2REGN == 9  ~ "Wales",
      BD2REGN == 10 ~ "Scotland",
      BD2REGN %in% c(-1, -2, 11, 12) ~ NA_character_,
      TRUE ~ NA_character_
    ),

    # --- Birth (BD1REGN) ---
    location_bd1 = case_when(
      BD1REGN == 1  ~ "North",
      BD1REGN == 2  ~ "Yorks and Humberside",
      BD1REGN == 3  ~ "East Midlands",
      BD1REGN == 4  ~ "East Anglia",
      BD1REGN == 5  ~ "South East",
      BD1REGN == 6  ~ "South West",
      BD1REGN == 7  ~ "West Midlands",
      BD1REGN == 8  ~ "North West",
      BD1REGN == 9  ~ "Wales",
      BD1REGN == 10 ~ "Scotland",
      BD1REGN %in% c(-1, -2, 11, 12) ~ NA_character_,
      TRUE ~ NA_character_
    ),

    # --- Prefer bd3, then bd2, then bd1 ---
    location = factor(
      coalesce(location_bd3, location_bd2, location_bd1),
      levels = c(
        "South East", "North", "Yorks and Humberside", "East Midlands", "East Anglia",
        "South West", "West Midlands", "North West",
        "Wales", "Scotland"
      )
    )
  )


### SOCIAL INDEX
df_process <- df_process %>%
  mutate(
    # ---- Age 10 (preferred) ----
    socialindex_bd3 = case_when(
      bd3psoc %in% c(-1, -2) ~ NA_character_,     # nk/ns, armed services → NA
      bd3psoc == 1 ~ "V unskilled",
      bd3psoc == 2 ~ "IV partly-skilled",
      bd3psoc == 3 ~ "III manual",
      bd3psoc == 4 ~ "III non-manual",
      bd3psoc == 5 ~ "II managerial/technical",
      bd3psoc == 6 ~ "I professional",
      TRUE ~ NA_character_
    ),

    # ---- Age 5 (fallback #1) ----
    socialindex_bd2 = case_when(
      BD2SOC %in% c(-1, 0) ~ NA_character_,       # info missing / student-voluntary → NA
      BD2SOC == 1 ~ "V unskilled",
      BD2SOC == 2 ~ "IV partly-skilled",
      BD2SOC == 3 ~ "III manual",
      BD2SOC == 4 ~ "III non-manual",
      BD2SOC == 5 ~ "II managerial/technical",
      BD2SOC == 6 ~ "I professional",
      TRUE ~ NA_character_
    ),

    # ---- Birth (fallback #2) ----
    socialindex_bd1 = case_when(
      BD1PSOC %in% c(-1) ~ NA_character_,         # nk/ns → NA
      BD1PSOC %in% c(1, 2) ~ NA_character_,       # single parent / other category → NA
      BD1PSOC == 3 ~ "V unskilled",
      BD1PSOC == 4 ~ "IV partly-skilled",
      BD1PSOC == 5 ~ "III manual",
      BD1PSOC == 6 ~ "III non-manual",
      BD1PSOC == 7 ~ "II managerial/technical",
      BD1PSOC == 8 ~ "I professional",
      TRUE ~ NA_character_
    ),

    # ---- Coalesce in priority order and make ordered factor ----
    socialindex = factor(
      coalesce(socialindex_bd3, socialindex_bd2, socialindex_bd1),
      levels = c("III manual", "V unskilled", "IV partly-skilled",
                 "III non-manual", "II managerial/technical", "I professional"),
      ordered = FALSE
    )
  )




### MATERNAL SMOKING
df_process <- df_process %>%
  mutate(
    maternal_smoking = case_when(
      e10.1 == 1 ~ "Ever",           # Has smoked
      e10.1 == 2 ~ "Never",          # Never in past 10 years
      e10.1 %in% c(-1, -2, -3, -4) ~ NA_character_,  # missing codes
      TRUE ~ NA_character_
    ),
    maternal_smoking = factor(maternal_smoking, levels = c("Never", "Ever"))
  )


### INCOME
df_process <- df_process %>%
  mutate(
    income = case_when(
      bd3inc %in% c(-1, 8) ~ "Refused/NA",   # include -1 and explicit Refused
      is.na(bd3inc)        ~ "Refused/NA",   # capture true NAs
      bd3inc %in% c(0, 1)  ~ "£200+",
      bd3inc == 2          ~ "£150-£199",
      bd3inc == 3          ~ "£100-£149",
      bd3inc %in% c(4, 5, 6)  ~ "£99 and under",
      TRUE ~ "Refused/NA"                     # safety net
    ),
    income = factor(
      income,
      levels = c(
        "£99 and under",
        "£100-£149",
        "£150-£199",
        "£200+",
        "Refused/NA"
      ),
      ordered = FALSE
    )
  )




In [ ]:
### PARENTAL EDUCATION 
df_process <- df_process %>%
  mutate(
    f_ed = case_when(
      c1.9  == 1 ~ 1,  # No qualifications
      c1.1  == 1 ~ 2,  # Trade apprentice
      c1.2  == 1 ~ 3,  # O Level
      c1.3  == 1 ~ 4,  # A Level
      c1.4  == 1 ~ 5,  # SRN
      c1.5  == 1 ~ 6,  # Cert Ed
      c1.6  == 1 ~ 7,  # Degree
      c1.7  == 1 | c1.8 > 0 ~ 8,  # Other / classified
      c1.11 == 1 ~ 9,  # Not known
      c1.10 == 1 ~ 10, # No male head
      TRUE ~ NA_real_
    ),
    f_ed = factor(f_ed, levels = 1:10,
                  labels = c("No qualifications", "Trade apprentice", "O Level", "A Level",
                             "SRN", "Cert Ed", "Degree", "Other/classified",
                             "Not known", "No male head")),
    
    m_ed = case_when(
      c1.20 == 1 ~ 1,  # No qualifications
      c1.12 == 1 ~ 2,  # Trade apprentice
      c1.13 == 1 ~ 3,  # O Level
      c1.14 == 1 ~ 4,  # A Level
      c1.15 == 1 ~ 5,  # SRN
      c1.16 == 1 ~ 6,  # Cert Ed
      c1.17 == 1 ~ 7,  # Degree
      c1.18 == 1 | c1.19 > 0 ~ 8,  # Other / classified
      c1.22 == 1 ~ 9,  # Not known
      c1.21 == 1 ~ 10, # No female head
      TRUE ~ NA_real_
    ),
    m_ed = factor(m_ed, levels = 1:10,
                  labels = c("No qualifications", "Trade apprentice", "O Level", "A Level",
                             "SRN", "Cert Ed", "Degree", "Other/classified",
                             "Not known", "No female head"))
  )

df_process <- df_process %>%
  # (keep your f_ed and m_ed construction as-is) %>%
  mutate(
    # --- Recode to numeric levels using REAL NA (key fix) ---
    f_ed_level_num = case_when(
      f_ed %in% "No qualifications"                      ~ 0L,
      f_ed %in% c("Trade apprentice", "O Level")         ~ 1L,
      f_ed %in% c("A Level", "SRN", "Cert Ed",
                  "Degree", "Other/classified")          ~ 2L,
      f_ed %in% c("Not known", "No male head")           ~ NA_integer_,
      TRUE                                               ~ NA_integer_
    ),
    m_ed_level_num = case_when(
      m_ed %in% "No qualifications"                      ~ 0L,
      m_ed %in% c("Trade apprentice", "O Level")         ~ 1L,
      m_ed %in% c("A Level", "SRN", "Cert Ed",
                  "Degree", "Other/classified")          ~ 2L,
      m_ed %in% c("Not known", "No female head")         ~ NA_integer_,
      TRUE                                               ~ NA_integer_
    ),

    # --- Highest of either parent; if both NA -> NA ---
    parent_num = if_else(
      is.na(f_ed_level_num) & is.na(m_ed_level_num),
      NA_integer_,
      pmax(f_ed_level_num, m_ed_level_num, na.rm = TRUE)
    ),

    # --- Turn back into factors (use true NA for missing) ---
    f_ed_level = factor(
      f_ed_level_num, levels = 0:2,
      labels = c("No quals", "Lower secondary", "Upper secondary/Higher education")
    ),
    m_ed_level = factor(
      m_ed_level_num, levels = 0:2,
      labels = c("No quals", "Lower secondary", "Upper secondary/Higher education")
    ),
    parent_ed  = factor(
      parent_num, levels = 0:2,
      labels = c("No quals", "Lower secondary", "Upper secondary/Higher education")
    )
  ) %>%
  select(-f_ed_level_num, -m_ed_level_num, -parent_num)

df_process <- df_process %>%
  mutate(
    f_ed_level = forcats::fct_na_value_to_level(f_ed_level, "NA"),
    m_ed_level = forcats::fct_na_value_to_level(m_ed_level, "NA"),
    parent_ed  = forcats::fct_na_value_to_level(parent_ed,  "Not Applicable")
  )

In [ ]:
### MATERNAL AGE AT BIRTH AND SIBLINGS
df_process <- df_process %>%
  mutate(
    # --- Child birth year (YY as stored) ---
    child_birth_year = a4a.4,

    # --- Mother's birth year from household roster (age 10 sweep) ---
    mother_birth_year = case_when(
      # --- In-house roster: natural mother only ---
      a4a.5  == 2 & a4a.8  >= 0 ~ a4a.8,
      a4a.9  == 2 & a4a.12 >= 0 ~ a4a.12,
      a4a.13 == 2 & a4a.16 >= 0 ~ a4a.16,
      a4a.17 == 2 & a4a.20 >= 0 ~ a4a.20,
      a4a.21 == 2 & a4a.24 >= 0 ~ a4a.24,
      a4a.25 == 2 & a4a.28 >= 0 ~ a4a.28,
      a4a.29 == 2 & a4a.32 >= 0 ~ a4a.32,
      a4a.33 == 2 & a4a.36 >= 0 ~ a4a.36,
      a4a.37 == 2 & a4a.40 >= 0 ~ a4a.40,
      # --- Not-in-house roster: natural mother only ---
      a4b.1  == 2 & a4b.4  >= 0 ~ a4b.4,
      a4b.6  == 2 & a4b.9  >= 0 ~ a4b.9,
      a4b.11 == 2 & a4b.14 >= 0 ~ a4b.14,
      a4b.16 == 2 & a4b.19 >= 0 ~ a4b.19,
      TRUE ~ NA_real_
    ),

    # --- 1) Preferred: roster-derived age at birth ---
    mother_age_at_birth_roster = ifelse(
      is.na(child_birth_year) | is.na(mother_birth_year),
      NA_real_,
      child_birth_year - mother_birth_year
    ),

    # --- 2) Fallback: birth sweep maternal age (BD1MAGE), negatives -> NA ---
    mother_age_at_birth_birth = ifelse(BD1MAGE < 0, NA_real_, as.numeric(BD1MAGE)),

    # --- 3) Fallback: age-5 sweep "Age of Mother" (e008) -> subtract 5 years; negatives -> NA ---
    mother_age_at_birth_age5 = case_when(
      e008 %in% c(-4, -3, -2, -1) ~ NA_real_,
      !is.na(e008) ~ as.numeric(e008) - 5,
      TRUE ~ NA_real_
    ),

    # --- Final: priority coalesce ---
    mother_age_at_birth = coalesce(mother_age_at_birth_roster,
                                   mother_age_at_birth_birth,
                                   mother_age_at_birth_age5),

      mother_age_at_birth = ifelse(
  mother_age_at_birth < 14 | mother_age_at_birth > 55,
  NA_real_,
  mother_age_at_birth
),
      
    # Count of older siblings (11 = elder brother, 12 = elder sister)
    older_siblings = rowSums(
      cbind(
        a4a.5  %in% c(11, 12),
        a4a.9  %in% c(11, 12),
        a4a.13 %in% c(11, 12),
        a4a.17 %in% c(11, 12),
        a4a.21 %in% c(11, 12),
        a4a.25 %in% c(11, 12),
        a4a.29 %in% c(11, 12),
        a4a.33 %in% c(11, 12),
        a4a.37 %in% c(11, 12)
      ),
      na.rm = TRUE
    ),

    # Count of younger siblings (13 = younger brother, 14 = younger sister)
    younger_siblings = rowSums(
      cbind(
        a4a.5  %in% c(13, 14),
        a4a.9  %in% c(13, 14),
        a4a.13 %in% c(13, 14),
        a4a.17 %in% c(13, 14),
        a4a.21 %in% c(13, 14),
        a4a.25 %in% c(13, 14),
        a4a.29 %in% c(13, 14),
        a4a.33 %in% c(13, 14),
        a4a.37 %in% c(13, 14)
      ),
      na.rm = TRUE
    )
  )


#### Outcomes

##### Blood biomarkers

In [ ]:
# ------------------------------------------------------------------------------
# BIOMARKERS CLEANING AND DERIVATION
#
# This section standardizes and interprets biomarker data collected at age 46.
#
# ▸ GENERAL CLEANING:
# - Applies to all blood biomarker variables:
#     "Refused" (-9), "Unsuitable" (-8), and "Not applicable" (-1) → set to NA
#
# ▸ BIOMARKERS CLEANED:
#   "B10CHOL", "B10HDL", "B10HBA1C", "B10IGF1", "B10RTIN", "B10TRIG",
#   "B10RBC", "B10HSCRP", "B10USCMG", "B10USCMM", "B10CMVAVC", "CS_COV_RESULT"
#
# ▸ CMV IgG (B10USCMG):
# - Interpreted as binary:
#     - 2 = Positive → 1
#     - 1 or 3 = Negative or Unequivocal → 0
#     - Others → NA
# - Recoded as factor: "Negative", "Positive"
#
# ▸ BLOOD SAMPLE OBTAINED (B10SAMPTAK):
# - 1 = "Yes", 2 = "No"
# - -1, -8, -9 = missing → NA
# - Recoded as factor: "No", "Yes"
#
# This processing ensures biomarkers are usable in models and summary tables,
# with consistent NA handling and meaningful factor levels.
# ------------------------------------------------------------------------------

#Clean up biomarkers

#Make anything that was "Refused" (-9), "Unsuitable" (-8), or "Not applicable" (-1) into NA. 
bloodbiomarkers = c(
  "B10CHOL", "B10HDL", "B10HBA1C", "B10IGF1", "B10RTIN", "B10TRIG", "B10RBC",
  "B10HSCRP", "B10USCMG", "B10USCMM", "B10CMVAVC"
)

# Replace -9, -8, -1 with NA across these variables
df_process[bloodbiomarkers] = lapply(df_process[bloodbiomarkers], function(x) {
  x[x %in% c(-9, -8, -1)] = NA
  return(x)
})

##Categorical variables
#CMV IgG
df_process$B10USCMG = ifelse(df_process$B10USCMG == 2, 1,
                              ifelse(df_process$B10USCMG == 1, 0,
                              ifelse(df_process$B10USCMG == 3, 0, NA)))##consider equivocal to be negative

df_process$B10USCMG = factor(df_process$B10USCMG,
                                     levels = c(0, 1),
                                     labels = c("Negative", "Positive"))


In [ ]:
##blood
df_process = df_process %>%
  rename(
    chol     = B10CHOL,
    hdl      = B10HDL,
    hba1c    = B10HBA1C,
    igf1     = B10IGF1,
    frtin    = B10RTIN,
    trig     = B10TRIG,
    rbc      = B10RBC,
    crp      = B10HSCRP,
    cmvigg   = B10USCMG,
    cmvigm   = B10USCMM,
    cmvav    = B10CMVAVC
  )

##### Outcomes

In [ ]:
# ------------------------------------------------------------------------------
# assign_condition(): Assigns EVER (1), NEVER (0), or NA for a health condition
# using age 46 (B10) as the primary source, and earlier sweeps only to support
# the detection of EVER cases.
#
# Input:
# - positive_matrix: columns indicating positive responses (TRUE/FALSE), with the
#   age 46 (B10)/latest indicator in the first column, followed by prior sweeps.
# - negative_matrix: same structure, where TRUE = explicit negative (e.g., 0/2)
#
# Logic:
# - Assign 1 (EVER) if:
#     ▸ The age 46 value (B10) is positive (TRUE), OR
#     ▸ Any earlier wave indicates a positive response
#
# - Assign 0 (NEVER) if:
#     ▸ The age 46 value (B10) is explicitly negative (TRUE in negative_matrix[,1])
#
# - Assign NA if:
#     ▸ Age 46 is missing/unknown, AND
#     ▸ No prior sweep indicates a positive response
#
# This ensures:
# - B10 determines "NEVER" classification exclusively
# - Prior waves help rescue missing B10 data to identify "EVER"
# - Ambiguous or missing responses default to NA
# ------------------------------------------------------------------------------



assign_condition <- function(df, name, positive_matrix, negative_matrix) {
  n <- nrow(df)
  result <- rep(NA, n)
  
  # Extract age 46 values — first column
  b10_pos <- positive_matrix[, 1]
  b10_neg <- negative_matrix[, 1]

  # Check for any positive prior (if more than 1 column)
  prior_pos <- if (ncol(positive_matrix) > 1) {
    rowSums(positive_matrix[, -1, drop = FALSE], na.rm = TRUE) > 0
  } else {
    rep(FALSE, n)
  }

  # Assign 1 (EVER): age 46 positive OR any prior positive
  result[b10_pos == TRUE | prior_pos] <- 1

  # Assign 0 (NEVER): only if age 46 explicitly says NO
  result[is.na(result) & b10_neg == TRUE] <- 0

  # Assign as factor
  df[[name]] <- factor(result, levels = c(0, 1), labels = c(0, 1))
  return(df)
}


# Diabetes
diabetes_pos <- with(df_process, cbind(
    B10KHPB03 == 1,#wave 10 age 46 must be first
  b960455 == 1,
  b960521 == 1,
  diab == 1,
  dl112m == 1,
  bd7hpb04 == 1,
  b8khpb03 == 1,
  B9KHPB04 == 1,
  B9INSULN == 1, 
  B9PKHP04 == 1,
  B10INSULN == 1,
  B10DIABD == 1,
  !is.na(dl1age) & !(dl1age %in% c(98, 99))
))

diabetes_neg <- with(df_process, cbind(
      B10KHPB03 == 0,
  b960455 %in% c(-2, 8),
  b960521 %in% c(-2, 8),
  diab %in% c(0, 2),
  dl112m == 2,
  bd7hpb04 == 0,
  b8khpb03 == 0,
  B9KHPB04 == 0,
  B9INSULN == 2,
  B9PKHP04 == 0,
  B10INSULN == 2,
  B10DIABD == 2,
  is.na(dl1age) | dl1age %in% c(98, 99)
))

df_process <- assign_condition(df_process, "diabetes", diabetes_pos, diabetes_neg)

# Hypertension
htn_pos <- with(df_process, cbind(
      B10KHPB10 == 1,
  downhibp == 1,
  bd7hpb11 == 1,
  b8khpb09 == 1,
  B9KHPB09 == 1,
  B9PKHP09 == 1,
  !is.na(bp1age) & !(bp1age %in% c(98, 99))
))
htn_neg <- with(df_process, cbind(
      B10KHPB10 == 0,
  downhibp %in% c(0, 2),
  bd7hpb11 == 0,
  b8khpb09 == 0,
  B9KHPB09 == 0,
  B9PKHP09 == 0,
  is.na(bp1age) | bp1age %in% c(98, 99)
))
df_process <- assign_condition(df_process, "hypertension", htn_pos, htn_neg)


In [ ]:
# ------------------------------------------------------------------------------
# DERIVED VARIABLES
#
# BMI:
# - Body Mass Index is calculated as weight (kg) / height^2 (m^2)
# - Uses B10MWEIGHT and B10HEIGHTCM (from age 46)
# - Missing if either variable is unavailable or invalid
# ------------------------------------------------------------------------------


# BMI
df_process$bmi <- with(df_process, ifelse(
  !is.na(B10HEIGHTCM) & B10HEIGHTCM > 0 &
  !is.na(B10MWEIGHT) & B10MWEIGHT > 0,
  
  B10MWEIGHT / ( (B10HEIGHTCM / 100)^2 ),  # BMI calculation
  
  NA  # If height or weight is missing/invalid
))

df_process$height <- with(df_process, ifelse(
    !is.na(B10HEIGHTCM) & B10HEIGHTCM > 0,

    B10HEIGHTCM,
    NA
))


# Waist/hip ratio
df_process$waisthip <- with(df_process, ifelse(
  !is.na(B10WAISTAV) & B10WAISTAV> 0 &
  !is.na(B10HIPAV) & B10HIPAV > 0,
  
  B10WAISTAV / (B10HIPAV),

  NA
))

## BLOOD PRESSURE
# need at least 2 valid (>0) readings
sys_vars <- c("B10BPSYSR1","B10BPSYSR2","B10BPSYSR3")
dia_vars <- c("B10BPDIAR1","B10BPDIAR2","B10BPDIAR3")

valid_sys <- df_process[sys_vars] > 0 & !is.na(df_process[sys_vars])
valid_dia <- df_process[dia_vars] > 0 & !is.na(df_process[dia_vars])

df_process$bp_sys <- ifelse(
  rowSums(valid_sys) >= 2,
  rowMeans(df_process[sys_vars], na.rm = TRUE),
  NA
)

df_process$bp_dia <- ifelse(
  rowSums(valid_dia) >= 2,
  rowMeans(df_process[dia_vars], na.rm = TRUE),
  NA
)


In [ ]:
#cognitive

#AGE 10

df_process <- df_process %>%
  mutate(
    # Standardised Edinburgh Reading Test score
    reading_score_std10 = case_when(
      bd3read == -1 ~ NA_real_,
      TRUE ~ as.numeric(bd3read)
    ),

    # Estimated reading age at age 10
    reading_age10 = case_when(
      bd3rdage == -1 ~ NA_real_,
      TRUE ~ as.numeric(bd3rdage)
    ),

    # Raw Edinburgh Reading Test score (if already derived bd3rread exists)
    reading_score_raw10 = case_when(
      bd3rread == -1 ~ NA_real_,
      TRUE ~ as.numeric(bd3rread)
    ),

    # Friendly Maths Test score
    maths_score10 = case_when(
      bd3maths == -1 ~ NA_real_,
      TRUE ~ as.numeric(bd3maths)
    )
  )


#Age 16
df_process <- df_process %>%
  mutate(
    # Arithmetic (out of 60)  — source: mathscore
    maths_score16 = ifelse(mathscore == -1, NA_real_, as.numeric(mathscore)),

    # Estimated reading age at 16 — source: BD4RDAGE
    reading_age16 = ifelse(BD4RDAGE == -1, NA_real_, as.numeric(BD4RDAGE)),

    # Matrices (out of 11) — source: SCR_M
    matrices_score16 = ifelse(SCR_M == -1, NA_real_, as.numeric(SCR_M)),

    # Total reading tests (out of 75) — source: SCRTOTAL
    reading_score_total16 = ifelse(SCRTOTAL == -1, NA_real_, as.numeric(SCRTOTAL)),

    # ---- Optional: percentages of max ----
    maths_score16_pct = ifelse(!is.na(maths_score16), maths_score16 / 60, NA_real_),
    matrices_score16_pct = ifelse(!is.na(matrices_score16), matrices_score16 / 11, NA_real_),
    reading_score_total16_pct = ifelse(!is.na(reading_score_total16), reading_score_total16 / 75, NA_real_)
  )
#age 35

df_process <- df_process %>%
  mutate(
      numeracy35 = ifelse(numall %in% c(-1), NA_real_, as.numeric(numall)),
      literacy35 = ifelse(litall37 %in% c(-1), NA_real_, as.numeric(litall37)),
    )

#Age 42

df_process <- df_process %>%
  mutate(
      vocab42 = ifelse(B9VSCORE %in% c(-1), NA_real_, as.numeric(B9VSCORE))
    )


#Age 46
df_process <- df_process %>%
  mutate(
    # Word list recall
    wordlist_immediate46 = ifelse(B10CFLISN %in% c(-9, -8, -1), NA_real_, as.numeric(B10CFLISN)),
    wordlist_delayed46   = ifelse(B10CFLISD %in% c(-9, -8, -1), NA_real_, as.numeric(B10CFLISD)),

    # Animal naming
    animal_naming46 = ifelse(B10CFANI %in% c(-9, -8, -1), NA_real_, as.numeric(B10CFANI)),

    # Letter cancellation
    letter_correct46 = ifelse(B10CFCOR %in% c(-9, -8, -1), NA_real_, as.numeric(B10CFCOR)),
    letter_missed46  = ifelse(B10CFMIS %in% c(-9, -8, -1), NA_real_, as.numeric(B10CFMIS)),
    letter_total46   = ifelse(B10CFTOT %in% c(-9, -8, -1), NA_real_, as.numeric(B10CFTOT)),
    letter_speed46   = ifelse(B10CFRC  %in% c(-9, -8, -1), NA_real_, as.numeric(B10CFRC))
  )



In [ ]:
head(df_process)

In [ ]:
df_process <- df_process %>%
  mutate(
    # convert missing codes to NA then numeric
    B9AGELE2  = ifelse(B9AGELE  %in% c(-9, -8, -1), NA, as.numeric(as.character(B9AGELE))),
    b7actagl2 = ifelse(b7actagl %in% c(-9, -8, -1), NA, as.numeric(as.character(b7actagl))),
    actagel2  = ifelse(actagel  %in% c(998, 999),   NA, as.numeric(as.character(actagel))),
    b9601322  = ifelse(b960132 %in% c(-8, -2, -1),  NA, as.numeric(as.character(b960132))),
    b9601292  = ifelse(b960129 %in% c(-8, -2, -1),  NA, as.numeric(as.character(b960129))),

    # pick first non-NA in preferred order: 39 -> 34/35 -> 29 -> 26a -> 26b
    adulteduc = coalesce(B9AGELE2, b7actagl2, actagel2, b9601322, b9601292),

  ) %>%
  select(-B9AGELE2, -b7actagl2, -actagel2, -b9601322, -b9601292)

df_process <- df_process %>%
  mutate(
    adulteduc = ifelse(adulteduc > 50, NA_real_, adulteduc)
  )

#### Clean

In [ ]:
head(df_process)

In [ ]:
##CLEAN AND SAVE DATAFRAME
clean_names = c(
    "BCSID", #id
    
    "chol", "hdl", "hba1c", "igf1", "frtin", "trig", "rbc", "crp", "cmvigg",#blood
    
    "measlesinfection", #exposure
    
    "sex", "location", "socialindex", 
    "maternal_smoking", "mother_age_at_birth", "ethnicity_binary", 
    "older_siblings", "younger_siblings", "income", "parent_ed", #covariates
    
    "diabetes", "hypertension", "bmi", "waisthip", "height","bp_sys", "bp_dia", #cardiometabolic outcomes

    "reading_score_raw10", "maths_score10",
    "maths_score16", "matrices_score16", "reading_score_total16", "literacy35", "numeracy35", "vocab42",
    "wordlist_immediate46", "wordlist_delayed46", "animal_naming46", 
    "letter_correct46", "letter_speed46", #cognitive

    "adulteduc"
)

In [ ]:
df_cleaned = df_process %>% select(all_of(clean_names))

summary(df_cleaned)

In [ ]:
covariates = c("sex", "location", "socialindex",
               "maternal_smoking", "mother_age_at_birth",  
               "older_siblings", "younger_siblings", "income", "parent_ed")

In [ ]:
df_full <- df_cleaned %>%
  filter(!is.na(measlesinfection))

df_analytic <- df_full %>%
  filter(complete.cases(across(all_of(covariates))))

nrow(df_full)
nrow(df_analytic)

summary(df_analytic$measlesinfection)

In [ ]:
# Complete sample
df_full <- df_cleaned %>%
  filter(!is.na(measlesinfection))

# Analytic sample (only complete covariate cases)
df_analytic <- df_full %>%
  filter(complete.cases(across(all_of(covariates))))

# Blood sample
bloodvars <- c("chol", "hdl", "hba1c", "igf1", "frtin", "trig", "rbc", "crp", "cmvigg")

df_blood <- df_analytic %>%
  filter(rowSums(!is.na(across(all_of(bloodvars)))) > 0)

In [ ]:
# Log transform CRP, FRTIN, and TRIG
df_analytic <- df_analytic %>%
  mutate(
    log_crp = log1p(crp),
    log_frtin = log1p(frtin),
    log_trig = log1p(trig),
    diab_hba1c = ifelse(diabetes == 1, hba1c + 40, hba1c)
  )

continuousblood = c("chol", "hdl", "hba1c", "igf1", "log_frtin", "log_trig", "rbc", "log_crp", "diab_hba1c")

In [ ]:
# Create standardized versions with _std suffix
df_analytic[paste0(continuousblood, "_std")] <- lapply(df_analytic[continuousblood], scale)

continuousblood_std = c("chol_std", "hdl_std", "hba1c_std", "igf1_std", "log_frtin_std", "log_trig_std", "rbc_std", "log_crp_std")

In [ ]:
label_map <- c(

  # ID
  "BCSID" = "BCSID",

  # Blood markers (raw and standardized/log-transformed)
  "chol"             = "Cholesterol",
  "hdl"              = "HDL",
  "hba1c"            = "HbA1c",
  "igf1"             = "IGF-1",
  "frtin"            = "Ferritin",
  "trig"             = "Triglycerides",
  "rbc"              = "Red Blood Cell Count",
  "crp"              = "C-Reactive Protein",
  "cmvigg"           = "CMV IgG",

  "log_frtin"        = "log(Ferritin)",
  "log_trig"         = "log(Triglycerides)",
  "log_crp"          = "log(CRP)",

  "chol_std"         = "Cholesterol",
  "hdl_std"          = "HDL",
  "hba1c_std"        = "HbA1c",
  "diab_hba1c_std"   = "HbA1c adjusted for diabetes",
  "igf1_std"         = "IGF-1",
  "log_frtin_std"    = "log(Ferritin)",
  "log_trig_std"     = "log(Triglycerides)",
  "rbc_std"          = "Red Blood Cell Count",
  "log_crp_std"      = "log(CRP)",

  # Exposure
  "measlesinfection" = "Measles Infection",

  # Covariates
  "sex"              = "Sex",
  "ethnicity"        = "Ethnicity",
  "ethnicity_binary" = "Ethnicity",
  "location"         = "Location",
  "socialindex"      = "Social Index (Categorical)",
  "older_siblings"   = "Older Siblings",
  "younger_siblings" = "Younger Siblings",
  "maternal_smoking" = "Maternal Smoking",
  "mother_age_at_birth"     = "Maternal Age",
  "income"           = "Income",

#"children_older_num", "children_younger_num", "socialclass", "income", "mother_edu", "father_edu"

  # Cardiometabolic outcomes
  "diabetes"         = "Diabetes",
  "hypertension"     = "Hypertension",
  "hypertension_bp"  = "Hypertension from BP",
  "bmi_z"              = "Body Mass Index (BMI)",
  "waisthip_z"         = "Waist/Hip Ratio",
  "bp_sys_z"           = "Systolic Blood Pressure",
  "bp_dia_z"           = "Diastolic Blood Pressure",
  "height_z" = "Height",


  # Cognitive Outcomes
  "reading_score_raw10_z" = "Age 10 Reading",
  "maths_score10_z"       = "Age 10 Maths",

  "maths_score16_z"       = "Age 16 Arithmetic",
  "matrices_score16_z"    = "Age 16 Matrices",
  "reading_score_total16_z" = "Age 16 Reading",

  "literacy35_z" = "Age 35 Literacy",
  "numeracy35_z" = "Age 35 Numeracy",
  "vocab42_z"   = "Age 42 Vocab",
    
  "wordlist_immediate46_z"  = "Age 46 Word List Recall (immediate)",
  "wordlist_delayed46_z"    = "Age 46 Word List Recall (delayed)",
  "animal_naming46_z"       = "Age 46 Verbal Fluency",
  "letter_correct46_z"      = "Age 46 Letter Cancellation (score)",
  "letter_speed46_z"        = "Age 46 Letter Cancellation (speed)",

  "adulteduc"              = "Educational Attainment"
    
)

categorical_outcomes <- c(
  "diabetes", "hypertension"
)

cognitive_continuous_adult <- c( 
  "wordlist_immediate46_z", "wordlist_delayed46_z",
  "animal_naming46_z", "letter_correct46_z", "letter_speed46_z","vocab42_z", "literacy35_z","numeracy35_z")

adult_educ <- c("adulteduc")

cognitive_continuous_child <- c(
  "reading_score_raw10_z", "maths_score10_z",
  "maths_score16_z", "matrices_score16_z", "reading_score_total16_z")

continuous_outcomes <- c("bmi_z", "waisthip_z", "bp_dia_z", "bp_sys_z")

continuous_biomarkers = c(continuousblood_std, continuous_outcomes, "cmvigg")

outcomes <- c(categorical_outcomes, cognitive_continuous_adult, adult_educ, cognitive_continuous_child, continuous_outcomes, continuous_biomarkers)

# ---- Outcome family map (used for main + IPW analyses) ----

family_map <- c(
  # Binary outcomes
  diabetes = "binomial",
  hypertension = "binomial",

  # Adult cognitive outcomes (continuous)
  vocab42_z   = "gaussian",
  literacy35_z = "gaussian",
  numeracy35_z = "gaussian",
  wordlist_immediate46_z = "gaussian",
  wordlist_delayed46_z   = "gaussian",
  animal_naming46_z      = "gaussian",
  letter_correct46_z     = "gaussian",
  letter_speed46_z       = "gaussian",

  # Adult education (continuous)
  adulteduc = "gaussian",

  # Childhood / adolescent cognitive outcomes (continuous)
  reading_score_raw10_z    = "gaussian",
  maths_score10_z          = "gaussian",
  maths_score16_z          = "gaussian",
  matrices_score16_z       = "gaussian",
  reading_score_total16_z  = "gaussian",

  # Anthropometric / BP outcomes (continuous)
  bmi_z       = "gaussian",
  waisthip_z  = "gaussian",
  bp_dia_z    = "gaussian",
  bp_sys_z    = "gaussian",

  cmvigg = "binomial",

  # Additional standardized biomarkers (continuous)
  chol_std       = "gaussian",
  hdl_std        = "gaussian",
  hba1c_std      = "gaussian",
  igf1_std       = "gaussian",
  log_frtin_std  = "gaussian",
  log_trig_std   = "gaussian",
  rbc_std        = "gaussian",
  log_crp_std    = "gaussian"
)



In [ ]:
df_analytic <- df_analytic %>%
  mutate(across(
    c(reading_score_raw10, maths_score10,
      maths_score16, reading_score_total16, matrices_score16, literacy35, numeracy35, vocab42,
      wordlist_immediate46, wordlist_delayed46, animal_naming46,
      letter_correct46, letter_speed46, bmi, waisthip, bp_dia, bp_sys),
    ~ scale(.)[,1], 
    .names = "{.col}_z"
  ))

In [ ]:
#### ggplot theme
my_custom_theme <- function(base_size = 12, base_family = "Helvetica") {
  theme_minimal(base_size = base_size, base_family = base_family) +
    theme(
      # Text
      plot.title = element_text(size = base_size + 4, face = "bold", hjust = 0.5),
      axis.title = element_text(size = base_size + 2),
      axis.text = element_text(size = base_size, color = "black"),
      axis.ticks = element_line(color = "black"),  # Optional: show ticks

      # Grid and Background
      panel.grid = element_blank(),
      panel.background = element_blank(),
      plot.background = element_blank(),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.8),

      # Legend
      legend.position = "bottom",
      legend.title = element_text(face = "bold"),
      legend.text = element_text(size = base_size),  # Optional

      # Facet strips
      strip.text = element_text(face = "bold", size = base_size),

      # Margin
     # plot.margin = margin(10, 10, 10, 10)  # Optional
    )
}


In [ ]:
covariates_reg = c("sex", "location", "socialindex",
               "maternal_smoking", "mother_age_at_birth",  
               "older_siblings", "younger_siblings", "income", "parent_ed")

In [ ]:
continuous_outcomes <- c("bmi_z", "waisthip_z", "bp_dia_z", "bp_sys_z")

continuous_biomarkers = c(continuousblood_std, continuous_outcomes)

## ANALYSES

### IPW

In [ ]:
# 1. Fit propensity score for measles infection
# ensure measlesinfection is a 0/1 indicator for modeling
df_ps <- df_analytic %>% mutate(.A = as.numeric(measlesinfection == "Yes")) %>%
  droplevels()  # <-- add this

ps_formula <- as.formula(paste(".A ~", paste(covariates_reg, collapse = " + ")))
ps_mod <- glm(ps_formula, data = df_ps, family = binomial)

df_ps$ps <- predict(ps_mod, type = "response")
pA <- mean(df_ps$.A, na.rm = TRUE)  # marginal probability of treatment

# 2. Stabilized weights
df_ps$sw <- ifelse(df_ps$.A == 1, pA / df_ps$ps, (1 - pA) / (1 - df_ps$ps))

# 3. Truncate extreme weights at 1st and 99th percentiles (adjustable)
q <- quantile(df_ps$sw, probs = c(0.01, 0.99), na.rm = TRUE)
df_ps$sw_trunc <- pmin(pmax(df_ps$sw, q[1]), q[2])

# Report basic diagnostics
cat("PS model AUC (approx via c-stat):\n")
if (requireNamespace("pROC", quietly = TRUE)) {
  print(pROC::auc(pROC::roc(df_ps$.A ~ df_ps$ps)))
} else print("Install pROC for AUC")

cat("Stabilized weights summary (raw):\n"); print(summary(df_ps$sw))
cat("Stabilized weights summary (truncated):\n"); print(summary(df_ps$sw_trunc))
Neff <- (sum(df_ps$sw_trunc, na.rm = TRUE)^2) / sum(df_ps$sw_trunc^2, na.rm = TRUE)
cat("Effective sample size (after truncation):", round(Neff, 1), "\n")

# Optional: balance check (SMD before / after)
bal_before <- bal.tab(
  ps_formula,
  data = df_ps,
  treat = df_ps$.A,
  estimand = "ATE",
  un = TRUE
)

bal_after <- bal.tab(
  ps_formula,
  data = df_ps,
  treat = df_ps$.A,
  weights = df_ps$sw_trunc,
  estimand = "ATE"
)

# Print summary (this never depends on column names)
cat("\nBalance summary BEFORE weighting:\n")
print(summary(bal_before))

cat("\nBalance summary AFTER weighting:\n")
print(summary(bal_after))

# Extract max absolute standardized mean difference (reviewer-friendly)
get_max_smd <- function(bal, col_pattern = "Diff") {
  bal_df  <- as.data.frame(bal$Balance)
  smd_cols <- grep(col_pattern, names(bal_df), value = TRUE)
  if (length(smd_cols) == 0) return(NA_real_)
  max(abs(bal_df[, smd_cols, drop = FALSE]), na.rm = TRUE)
}

cat("\nMax |SMD| BEFORE weighting:", round(get_max_smd(bal_before, "Diff.Un"),  3), "\n")
cat("Max |SMD| AFTER weighting:",  round(get_max_smd(bal_after,  "Diff.Adj"), 3), "\n")

# ---- End balance diagnostics ----
# ── Compile diagnostics into appendix table ───────────────────────────────────

# 1. AUC
auc_val <- as.numeric(pROC::auc(pROC::roc(response  = df_ps$.A,
                                            predictor = df_ps$ps)))

# 2. PS distribution by group
ps_summary <- df_ps %>%
  mutate(group = ifelse(.A == 1, "Measles (Yes)", "Measles (No)")) %>%
  group_by(group) %>%
  summarise(
    ps_mean = round(mean(ps, na.rm = TRUE), 3),
    ps_sd   = round(sd(ps,   na.rm = TRUE), 3),
    ps_min  = round(min(ps,  na.rm = TRUE), 3),
    ps_max  = round(max(ps,  na.rm = TRUE), 3),
    .groups = "drop"
  ) %>%
  mutate(ps_summary = paste0(ps_mean, " (SD: ", ps_sd, ", range: ", ps_min, "-", ps_max, ")")) %>%
  select(group, ps_summary)

# 3. Weight summaries
sw_raw_s   <- summary(df_ps$sw)
sw_trunc_s <- summary(df_ps$sw_trunc)

pct_truncated <- round(mean(df_ps$sw != df_ps$sw_trunc, na.rm = TRUE) * 100, 1)
Neff          <- round((sum(df_ps$sw_trunc)^2) / sum(df_ps$sw_trunc^2), 1)

# 4. SMDs
max_smd_before <- round(get_max_smd(bal_before, "Diff.Un"),  3)
max_smd_after  <- round(get_max_smd(bal_after,  "Diff.Adj"), 3)

# Count covariates above threshold before and after
bal_df_before <- as.data.frame(bal_before$Balance)
bal_df_after  <- as.data.frame(bal_after$Balance)

n_imbalanced_before <- sum(abs(bal_df_before$Diff.Un)  > 0.1, na.rm = TRUE)
n_imbalanced_after  <- sum(abs(bal_df_after$Diff.Adj)  > 0.1, na.rm = TRUE)
n_covariates        <- nrow(bal_df_before)

# ── Build summary table ───────────────────────────────────────────────────────
diag_table <- tibble(
  Diagnostic = c(
    # Sample
    "Analytic sample (N)",
    "Measles exposed (n, %)",
    "Measles unexposed (n, %)",
    # PS model
    "PS model",
    "PS model AUC (C-statistic)",
    paste0("PS mean (SD) — Measles Yes"),
    paste0("PS mean (SD) — Measles No"),
    # Weights
    "Stabilized weight — Mean (raw)",
    "Stabilized weight — Median (raw)",
    "Stabilized weight — Min (raw)",
    "Stabilized weight — Max (raw)",
    "Truncation percentiles",
    "Stabilized weight — Mean (truncated)",
    "Stabilized weight — Median (truncated)",
    "Stabilized weight — Min (truncated)",
    "Stabilized weight — Max (truncated)",
    "% observations truncated",
    "Effective sample size (after truncation)",
    # Balance
    "Max |SMD| before weighting",
    "Max |SMD| after weighting",
    paste0("N covariates |SMD| > 0.1 before weighting (of ", n_covariates, ")"),
    paste0("N covariates |SMD| > 0.1 after weighting (of ",  n_covariates, ")")
  ),
  Value = c(
    # Sample
    nrow(df_ps),
    paste0(sum(df_ps$.A == 1), " (",
           round(mean(df_ps$.A == 1) * 100, 1), "%)"),
    paste0(sum(df_ps$.A == 0), " (",
           round(mean(df_ps$.A == 0) * 100, 1), "%)"),
    # PS model
    paste(covariates_reg, collapse = ", "),
    round(auc_val, 3),
    ps_summary$ps_summary[ps_summary$group == "Measles (Yes)"],
    ps_summary$ps_summary[ps_summary$group == "Measles (No)"],
    # Weights raw
    round(sw_raw_s["Mean"],   3),
    round(sw_raw_s["Median"], 3),
    round(sw_raw_s["Min."],   3),
    round(sw_raw_s["Max."],   3),
    "1st and 99th percentiles",
    round(sw_trunc_s["Mean"],   3),
    round(sw_trunc_s["Median"], 3),
    round(sw_trunc_s["Min."],   3),
    round(sw_trunc_s["Max."],   3),
    paste0(pct_truncated, "%"),
    Neff,
    # Balance
    max_smd_before,
    max_smd_after,
    n_imbalanced_before,
    n_imbalanced_after
  )
)

print(diag_table)
write.csv(diag_table, "appendix_ipw_diagnostics.csv", row.names = FALSE)



In [ ]:
ps_coef_table <- broom::tidy(ps_mod, conf.int = TRUE, exponentiate = TRUE) %>%
  mutate(
    OR       = round(estimate, 3),
    conf.low  = round(conf.low, 3),
    conf.high = round(conf.high, 3),
    p.value  = round(p.value, 4),
    CI       = paste0(conf.low, " - ", conf.high)
  ) %>%
  select(term, OR, CI, p.value)

# Add stars for convenience
ps_coef_table <- ps_coef_table %>%
  mutate(sig = case_when(
    p.value < 0.001 ~ "***",
    p.value < 0.01  ~ "**",
    p.value < 0.05  ~ "*",
    p.value < 0.1   ~ ".",
    TRUE            ~ ""
  ))

print(ps_coef_table)
write.csv(ps_coef_table, "ps_model_coefficients.csv", row.names = FALSE)

In [ ]:
# Get balance object with both un-weighted and weighted SMDs
bal_full <- bal.tab(
  ps_formula,
  data     = df_ps,
  treat    = df_ps$.A,
  weights  = df_ps$sw_trunc,
  estimand = "ATE",
  un       = TRUE,
  disp     = c("means", "sds")
)

# Extract and format
bal_df <- as.data.frame(bal_full$Balance) %>%
  tibble::rownames_to_column("covariate") %>%
  select(
    covariate,
    mean_unweighted_control   = starts_with("M.0.Un"),
    mean_unweighted_treated   = starts_with("M.1.Un"),
    smd_before                = Diff.Un,
    mean_weighted_control     = starts_with("M.0.Adj"),
    mean_weighted_treated     = starts_with("M.1.Adj"),
    smd_after                 = Diff.Adj
  ) %>%
  mutate(
    across(where(is.numeric), ~round(.x, 3)),
    balanced = ifelse(abs(smd_after) < 0.1, "Yes", "No")
  )

print(bal_df)
write.csv(bal_df, "balance_table.csv", row.names = FALSE)

# Love plot to accompany the table
love.plot(
  bal_full,
  threshold  = 0.1,
  abs        = TRUE,
  var.order  = "unadjusted",
  colors     = c("red", "darkgreen"),
  shapes     = c("circle", "square"),
  title      = "Covariate Balance Before and After IPW",
  labels     = TRUE
)

In [ ]:
# ── Attrition comparison table ────────────────────────────────────────────────
# Define covariates to compare
covariates <- c("sex", "socialindex", "parent_ed", "income", 
                "older_siblings", "younger_siblings", "mother_age_at_birth", 
                "maternal_smoking", "location")

# Define outcomes and their sample indicators
# For each outcome, create a binary "included" flag
outcome_vars <- c(
  "reading_score_raw10_z", "maths_score10_z",
  "reading_score_total16_z", "maths_score16_z", "matrices_score16_z",
  "literacy35_z", "numeracy35_z","vocab42_z",
  "wordlist_immediate46_z", "wordlist_delayed46_z",
  "animal_naming46_z", "letter_correct46_z", "letter_speed46_z",
  "adulteduc",
  "hypertension", "diabetes",
  "chol_std", "hdl_std", "hba1c_std", "igf1_std",
  "log_frtin_std", "log_trig_std", "rbc_std",
  "log_crp_std", "bmi_z", "waisthip_z",
  "bp_sys_z", "bp_dia_z", "cmvigg"
)

# Readable outcome labels
outcome_labels <- c(
  "Age 10 reading", "Age 10 maths",
  "Age 16 reading", "Age 16 maths", "Age 16 matrices", "Age 35 Literacy", "Age 35 Numeracy", "Age 42 Vocab",
  "Age 46 word list (immediate)", "Age 46 word list (delayed)",
  "Age 46 verbal fluency", "Age 46 letter cancellation (score)", 
  "Age 46 letter cancellation (speed)",
  "School leaving age",
  "Hypertension", "Diabetes",
  "Cholesterol", "HDL", "HbA1c", "IGF-1",
  "Ferritin", "Triglycerides", "Red blood cell count",
  "CRP", "BMI", "Waist-hip ratio",
  "Systolic BP", "Diastolic BP", "CMV IgG"
)

# ── Function to compute SMD for a single covariate ───────────────────────────
compute_smd <- function(df, var, included_flag) {
  x_in  <- df[[var]][df[[included_flag]] == 1]
  x_out <- df[[var]][df[[included_flag]] == 0]
  
  if (is.numeric(df[[var]])) {
    smd <- (mean(x_in, na.rm = TRUE) - mean(x_out, na.rm = TRUE)) /
      sqrt((var(x_in, na.rm = TRUE) + var(x_out, na.rm = TRUE)) / 2)
  } else {
    # For categorical: use proportion in most common category
    tbl_in  <- prop.table(table(x_in))
    tbl_out <- prop.table(table(x_out))
    cats    <- union(names(tbl_in), names(tbl_out))
    p_in    <- mean(tbl_in[cats], na.rm = TRUE)
    p_out   <- mean(tbl_out[cats], na.rm = TRUE)
    smd     <- (p_in - p_out) /
      sqrt((p_in * (1 - p_in) + p_out * (1 - p_out)) / 2)
  }
  round(smd, 3)
}

# ── Build attrition table ─────────────────────────────────────────────────────
attrition_rows <- list()

for (i in seq_along(outcome_vars)) {
  ov  <- outcome_vars[i]
  lab <- outcome_labels[i]
  
  # Create included flag in df_ps
  df_ps[[".included"]] <- as.integer(!is.na(df_ps[[ov]]))
  
  n_included <- sum(df_ps$.included == 1, na.rm = TRUE)
  n_excluded <- sum(df_ps$.included == 0, na.rm = TRUE)
  
  # SMD for each covariate
  smds <- sapply(covariates, function(cv) {
    tryCatch(compute_smd(df_ps, cv, ".included"), error = function(e) NA)
  })
  
  max_smd <- round(max(abs(smds), na.rm = TRUE), 3)
  
  attrition_rows[[i]] <- tibble(
    outcome       = lab,
    n_included    = n_included,
    n_excluded    = n_excluded,
    pct_excluded  = round(100 * n_excluded / nrow(df_ps), 1),
    max_abs_smd   = max_smd,
    flag          = ifelse(max_smd > 0.1, "Yes", "No")
  )
}

attrition_table <- bind_rows(attrition_rows)

# ── Save ──────────────────────────────────────────────────────────────────────
write.csv(attrition_table, "appendix_attrition_comparison.csv", 
          row.names = FALSE)

print(attrition_table)

In [ ]:
summary(df_ps)

In [ ]:
# Run weighted outcome models
ipw_res_list <- list()
for (y in outcomes) {
  fam <- family_map[[y]]

    fml <- as.formula(paste(y, "~ measlesinfection"))
    df_use <- df_ps %>% filter(!is.na(as.vector(.data[[y]])), !is.na(sw_trunc))

  if (fam == "binomial") {
    fit_w <- glm(fml, data = df_use, family = binomial, weights = sw_trunc)
    vcov_w <- sandwich::vcovHC(fit_w, type = "HC3")
    coef_est <- coef(fit_w)
    se_rob <- sqrt(diag(vcov_w))
    z <- coef_est / se_rob
    pvals <- 2 * pnorm(-abs(z))
    ci_l <- exp(coef_est - qnorm(0.975) * se_rob)
    ci_u <- exp(coef_est + qnorm(0.975) * se_rob)
    res_row <- tibble(term = names(coef_est),
                      estimate = exp(coef_est),
                      std.error = se_rob,
                      statistic = z,
                      p.value = pvals,
                      conf.low = ci_l,
                      conf.high = ci_u,
                      model = "IPW", outcome = y,
                      family = "binomial",
                      n = nrow(df_use))
  } else {
    fit_w <- lm(fml, data = df_use, weights = sw_trunc)
    vcov_w <- sandwich::vcovHC(fit_w, type = "HC3")
    coef_est <- coef(fit_w)
    se_rob <- sqrt(diag(vcov_w))
    tstat <- coef_est / se_rob
    pvals <- 2 * pt(-abs(tstat), df = fit_w$df.residual)
    ci_l <- coef_est - qt(0.975, df = fit_w$df.residual) * se_rob
    ci_u <- coef_est + qt(0.975, df = fit_w$df.residual) * se_rob
    res_row <- tibble(term = names(coef_est),
                      estimate = coef_est,
                      std.error = se_rob,
                      statistic = tstat,
                      p.value = pvals,
                      conf.low = ci_l,
                      conf.high = ci_u,
                      model = "IPW", outcome = y,
                      family = "continuous",
                      n = nrow(df_use))
  }
  ipw_res_list[[y]] <- res_row
}

ipw_results <- bind_rows(ipw_res_list) %>%
  filter(grepl("measlesinfection", term))

In [ ]:
ipw_results

### CRUDE ASSOCIATIONS

In [ ]:
crude_res_list <- list()
for (y in outcomes) {
  fam <- family_map[[y]]
  fml <- as.formula(paste(y, "~ measlesinfection"))
  df_use <- df_ps %>% filter(!is.na(.data[[y]]), !is.na(sw_trunc))
  
  if (fam == "binomial") {
    fit_c <- glm(fml, data = df_use, family = binomial)
    coef_est <- coef(fit_c)
    se <- sqrt(diag(vcov(fit_c)))
    z <- coef_est / se
    crude_res_list[[y]] <- tibble(
      outcome  = y,
      estimate = exp(coef_est),
      conf.low  = exp(coef_est - qnorm(0.975) * se),
      conf.high = exp(coef_est + qnorm(0.975) * se),
      p.value  = 2 * pnorm(-abs(z)),
      family   = "binomial",
        n         = nrow(df_use)
    ) %>% filter(grepl("measlesinfection", term <- names(coef_est)))
  } else {
    fit_c <- lm(fml, data = df_use)
    coef_est <- coef(fit_c)
    se <- sqrt(diag(vcov(fit_c)))
    tstat <- coef_est / se
    crude_res_list[[y]] <- tibble(
      outcome  = y,
      estimate = coef_est,
      conf.low  = coef_est - qt(0.975, df = fit_c$df.residual) * se,
      conf.high = coef_est + qt(0.975, df = fit_c$df.residual) * se,
      p.value  = 2 * pt(-abs(tstat), df = fit_c$df.residual),
      family   = "continuous",
        n         = nrow(df_use)
    ) %>% filter(grepl("measlesinfection", term <- names(coef_est)))
  }
}

crude_results <- bind_rows(crude_res_list)

In [ ]:
crude_results

### MERGE CRUDE AND IPW

In [ ]:
# --- Merge crude and IPW ---
comparison_table <- crude_results %>%
  select(outcome, family,
         crude_est = estimate,
         crude_low = conf.low,
         crude_high = conf.high,
         crude_p   = p.value,
         n_crude   = n) %>%
  left_join(
    ipw_results %>%
      select(outcome,
             ipw_est   = estimate,
             ipw_low   = conf.low,
             ipw_high  = conf.high,
             ipw_p     = p.value,
             n_ipw     = n),
    by = "outcome"
  ) %>%
  mutate(
    effect_measure = ifelse(family == "binomial", "OR", "Beta"),
    across(c(crude_est, crude_low, crude_high,
             ipw_est,   ipw_low,   ipw_high), ~round(.x, 3)),
    crude_ci = paste0(crude_low, " - ", crude_high),
    ipw_ci   = paste0(ipw_low,   " - ", ipw_high),
    
    # ── A priori outcome groups — EDIT to match your variable names ───────────
    outcome_group = case_when(
      outcome %in% c("hypertension",
                    "diabetes")        ~ "cardiometabolic",

      outcome %in% c("cmvigg",
                     "bmi_z",
                     "bp_dia_z",
                     "bp_sys_z",
                     "chol_std",
                     "hba1c_std",
                     "hdl_std",
                     "igf1_std",
                     "log_crp_std",
                     "log_frtin_std",
                     "log_trig_std",
                     "rbc_std",
                     "waisthip_z")        ~ "biomarker",
        
      outcome %in% c("animal_naming46_z",
                     "letter_correct46_z",
                     "letter_speed46_z",
                     "maths_score10_z",
                     "maths_score16_z",
                     "matrices_score16_z",
                     "reading_score_raw10_z",
                     "reading_score_total16_z",
                     "wordlist_delayed46_z",
                     "wordlist_immediate46_z", "vocab42_z","literacy35_z","numeracy35_z") ~ "cognitive",

      TRUE ~ "other"   # catches anything not yet assigned — check this is empty
    )
  ) %>%
  # ── BH adjustment within each a priori group ─────────────────────────────────
  group_by(outcome_group) %>%
  mutate(
    crude_p_fdr = signif(p.adjust(crude_p, method = "BH"), digits = 4),
    ipw_p_fdr   = signif(p.adjust(ipw_p,   method = "BH"), digits = 4)
  ) %>%
  ungroup() %>%
  # ── Round raw p after FDR so both are consistent ──────────────────────────────
  mutate(across(c(crude_p, ipw_p), ~signif(.x, digits = 4))) %>%
  # ── Order rows by group then outcome ─────────────────────────────────────────
  arrange(outcome_group, family, outcome) %>%
  select(outcome_group, outcome, effect_measure, family,
         n_crude, crude_est, crude_ci, crude_low, crude_high, crude_p, crude_p_fdr,
         n_ipw,   ipw_est,   ipw_ci,   ipw_low,   ipw_high,   ipw_p,   ipw_p_fdr)

# ── Sanity check: anything landing in "other"? ────────────────────────────────
other_check <- comparison_table %>% filter(outcome_group == "other")
if (nrow(other_check) > 0) {
  cat("WARNING: these outcomes were not assigned to a group:\n")
  print(other_check$outcome)
} else {
  cat("All outcomes assigned to a group.\n")
}

print(comparison_table)
write.csv(comparison_table, "crude_vs_ipw_table.csv", row.names = FALSE)

### TABLE 1

In [ ]:
# --- Weighted Table 1 using IPW weights ---
# Create survey design object with truncated stabilized weights
df_weighted <- df_ps %>%
  filter(!is.na(sw_trunc)) %>%
  select(all_of(covariates), measlesinfection, sw_trunc)

svy_design <- svydesign(
  ids     = ~1,
  weights = ~sw_trunc,
  data    = df_weighted
)

# --- Unweighted Table 1 (your original, kept for comparison) ---
table1_unweighted <- df_analytic %>%
  select(all_of(covariates), measlesinfection) %>%
  tbl_summary(
    by         = measlesinfection,
    statistic  = all_continuous() ~ "{mean} ({sd})",
    type       = list(older_siblings ~ "continuous",
                      younger_siblings ~ "continuous"),
    missing    = "ifany"
  ) %>%
  add_overall() %>%
  add_p() %>%
  modify_header(label = "**Characteristic**") %>%
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Measles Status**") %>%
  bold_labels() %>%
  modify_caption("**Table 1A. Analytic Sample (Unweighted)**")

# --- Weighted Table 1 ---
table1_weighted <- tbl_svysummary(
  data       = svy_design,
  by         = measlesinfection,
  include    = all_of(covariates),
  statistic  = all_continuous() ~ "{mean} ({sd})",
  type       = list(older_siblings ~ "continuous",
                    younger_siblings ~ "continuous"),
  missing    = "ifany"
) %>%
  add_overall() %>%
  add_p() %>%
  modify_header(label = "**Characteristic**") %>%
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Measles Status**") %>%
  bold_labels() %>%
  modify_caption("**Table 1B. Analytic Sample (IPW Weighted)**")

# --- Merge into a single side-by-side table ---
table1_combined <- tbl_merge(
  tbls          = list(table1_unweighted, table1_weighted),
  tab_spanner   = c("**Unweighted**", "**IPW Weighted**")
) %>%
  modify_caption("**Table 1. Covariate Distribution Before and After IPW**")

# --- Print ---

# --- Save combined ---
table1_combined %>%
  as_gt() %>%
  gtsave("table1_combined.html")

# Also save individual tables in case you need them separately
table1_unweighted %>%
  as_gt() %>%
  gtsave("table1_unweighted.html")

table1_weighted %>%
  as_gt() %>%
  gtsave("table1_weighted.html")

# CSV versions
write.csv(as.data.frame(table1_unweighted), "table1_unweighted.csv", row.names = FALSE)
write.csv(as.data.frame(table1_weighted),   "table1_weighted.csv",   row.names = FALSE)
write.csv(as.data.frame(table1_combined),   "table1_combined.csv",   row.names = FALSE)

In [ ]:
# Check distributions first
hist(df_ps$older_siblings)
hist(df_ps$younger_siblings)
hist(df_ps$mother_age_at_birth)

# Wilcoxon rank-sum test (non-parametric, appropriate for count/skewed variables)
wilcox.test(older_siblings ~ measlesinfection, data = df_ps)
wilcox.test(younger_siblings ~ measlesinfection, data = df_ps)
wilcox.test(mother_age_at_birth ~ measlesinfection, data = df_ps)

# T test
t.test(older_siblings ~ measlesinfection, data = df_ps)
t.test(younger_siblings ~ measlesinfection, data = df_ps)
t.test(mother_age_at_birth ~ measlesinfection, data = df_ps)

### SEX STRATIFIED

##### Test for interaction

In [ ]:
interaction_pvals <- list()

for (y in outcomes) {
  fam     <- family_map[[y]]
  fml_int <- as.formula(paste(y, "~ measlesinfection * sex"))
  df_use  <- df_ps %>%
    filter(!is.na(.data[[y]]), !is.na(sw_trunc), !is.na(sex)) %>%
    droplevels()

  if (fam == "binomial") {
    fit_int <- glm(fml_int, data = df_use, family = binomial, weights = sw_trunc)
  } else {
    fit_int <- lm(fml_int, data = df_use, weights = sw_trunc)
  }

  vcov_int <- sandwich::vcovHC(fit_int, type = "HC3")
  se_int   <- sqrt(diag(vcov_int))
  coef_int <- coef(fit_int)
  z_int    <- coef_int / se_int
  p_int    <- 2 * pnorm(-abs(z_int))

  int_term <- grep("measlesinfection.*sex|sex.*measlesinfection",
                   names(coef_int), value = TRUE)

  if (length(int_term) > 0) {
    ci_l <- coef_int[int_term] - qnorm(0.975) * se_int[int_term]
    ci_u <- coef_int[int_term] + qnorm(0.975) * se_int[int_term]
    
    # Exponentiate for binomial
    if (fam == "binomial") {
      est  <- exp(coef_int[int_term])
      ci_l <- exp(ci_l)
      ci_u <- exp(ci_u)
    } else {
      est  <- coef_int[int_term]
    }
    
    interaction_pvals[[y]] <- tibble(
      outcome        = y,
      family         = fam,
      int_term       = int_term,
      int_estimate   = est,
      int_ci_low     = ci_l,
      int_ci_high    = ci_u,
      int_p          = round(p_int[int_term], 4)
    )
  }
}

interaction_results <- bind_rows(interaction_pvals) %>%
  mutate(effect_measure = ifelse(family == "binomial", "OR", "Beta"),
         significant    = ifelse(int_p < 0.05, "Yes", "No"))

print(interaction_results)
write.csv(interaction_results, "interaction_by_sex.csv", row.names = FALSE)

# ── Quick summary of significant interactions ─────────────────────────────────
cat("\nOutcomes with significant sex interaction (p < 0.05):\n")
sig_int <- interaction_results %>% filter(significant == "Yes")
if (nrow(sig_int) == 0) {
  cat("  None\n")
} else {
  print(sig_int %>% select(outcome, effect_measure, int_p))
}

#### No interaction found

## COG STRATIFIED

In [ ]:
# ── Sensitivity analysis: stratified cognitive outcomes ───────────────────────

cognitive_outcomes <- c(
  "reading_score_raw10_z", "maths_score10_z",
  "reading_score_total16_z", "maths_score16_z", "matrices_score16_z","literacy35_z", "numeracy35_z","vocab42_z",
  "wordlist_immediate46_z", "wordlist_delayed46_z",
  "animal_naming46_z", "letter_correct46_z", "letter_speed46_z","adulteduc"
)

# Stratification variables and their levels (excluding refused/NA)
strat_vars <- list(
  socialindex = list(
    var = "socialindex",
    levels = c("V unskilled", "IV partly-skilled", "III manual",
               "III non-manual", "II managerial/technical", "I professional"),
    label = "Parental social index"
  ),
  income = list(
    var = "income",
    levels = c("£99 and under", "£100-£149", "£150-£199", "£200+"),
    label = "Parental income"
  ),
  parent_ed = list(
    var = "parent_ed",
    levels = c("No quals", "Lower secondary",
               "Upper secondary/Higher education"),
    label = "Parental education"
  )
)

# ── Run stratified models ─────────────────────────────────────────────────────
strat_results <- list()

for (sv in names(strat_vars)) {
  sv_info   <- strat_vars[[sv]]
  sv_var    <- sv_info$var
  sv_levels <- sv_info$levels

  rows <- list()
  for (lv in sv_levels) {
    df_strat <- df_ps %>% filter(.data[[sv_var]] == lv)

    for (y in cognitive_outcomes) {
      fam <- family_map[[y]]

      if (y %in% adult_outcomes) {
        fml    <- as.formula(paste(y, "~ measlesinfection"))
        df_use <- df_strat %>%
          filter(!is.na(.data[[y]]), !is.na(sw_trunc), !is.na(adulteduc))
      } else {
        fml    <- as.formula(paste(y, "~ measlesinfection"))
        df_use <- df_strat %>%
          filter(!is.na(.data[[y]]), !is.na(sw_trunc))
      }

      if (nrow(df_use) < 30) next  # skip strata too small to model

      tryCatch({
        if (fam == "binomial") {
          fit_w  <- glm(fml, data = df_use, family = binomial, weights = sw_trunc)
          vcov_w <- sandwich::vcovHC(fit_w, type = "HC3")
          coef_est <- coef(fit_w)
          se_rob   <- sqrt(diag(vcov_w))
          z        <- coef_est / se_rob
          pvals    <- 2 * pnorm(-abs(z))
          ci_l     <- exp(coef_est - qnorm(0.975) * se_rob)
          ci_u     <- exp(coef_est + qnorm(0.975) * se_rob)
          est      <- exp(coef_est)
        } else {
          fit_w  <- lm(fml, data = df_use, weights = sw_trunc)
          vcov_w <- sandwich::vcovHC(fit_w, type = "HC3")
          coef_est <- coef(fit_w)
          se_rob   <- sqrt(diag(vcov_w))
          tstat    <- coef_est / se_rob
          pvals    <- 2 * pt(-abs(tstat), df = fit_w$df.residual)
          ci_l     <- coef_est - qt(0.975, df = fit_w$df.residual) * se_rob
          ci_u     <- coef_est + qt(0.975, df = fit_w$df.residual) * se_rob
          est      <- coef_est
        }

        idx <- grep("measlesinfection", names(coef_est))
        rows[[length(rows) + 1]] <- tibble(
          stratum_var   = sv_info$label,
          stratum_level = lv,
          outcome       = y,
          outcome_label = label_map[y],
          estimate      = est[idx],
          conf.low      = ci_l[idx],
          conf.high     = ci_u[idx],
          p.value       = pvals[idx],
          n             = nrow(df_use)
        )
      }, error = function(e) {
        message("Skipping ", y, " in stratum ", lv, ": ", e$message)
      })
    }
  }
  strat_results[[sv]] <- bind_rows(rows)
}

# ── Format tables ─────────────────────────────────────────────────────────────
format_strat_table <- function(df) {
  df %>%
    mutate(
      est_fmt = ifelse(grepl("OR", outcome) | outcome %in% c("hypertension","diabetes","cmvigg"),
                       sprintf("%.3f", estimate),
                       sprintf("%.3f", estimate)),
      ci_fmt  = sprintf("(%.3f, %.3f)", conf.low, conf.high),
      p_fmt   = ifelse(p.value < 0.001, "<0.001", sprintf("%.3f", p.value))
    ) %>%
    select(stratum_level, outcome_label, n, est_fmt, ci_fmt, p_fmt) %>%
    rename(
      Stratum   = stratum_level,
      Outcome   = outcome_label,
      N         = n,
      Estimate  = est_fmt,
      `95% CI`  = ci_fmt,
      `P value` = p_fmt
    )
}

table_socialindex <- format_strat_table(strat_results$socialindex)
table_income      <- format_strat_table(strat_results$income)
table_parent_ed   <- format_strat_table(strat_results$parent_ed)

# Save tables
write.csv(table_socialindex, "strat_table_socialindex.csv", row.names = FALSE)
write.csv(table_income,      "strat_table_income.csv",      row.names = FALSE)
write.csv(table_parent_ed,   "strat_table_parent_ed.csv",   row.names = FALSE)

print(table_socialindex)
print(table_income)
print(table_parent_ed)

# ── Forest plots ──────────────────────────────────────────────────────────────
make_strat_forest <- function(df, title, x_limits = c(-1, 1),
                               x_breaks = seq(-1, 1, 0.25)) {
  
  # Fixed outcome order (top to bottom as in figure)
  outcome_order <- c(
    "Age 46 Letter Cancellation (speed)",
    "Age 46 Letter Cancellation (score)",
    "Age 46 Verbal Fluency",
    "Age 46 Word List Recall (delayed)",
    "Age 46 Word List Recall (immediate)",
    "Age 42 Vocab",
    "Age 35 Numeracy",
    "Age 35 Literacy",
    "Age 16 Reading",
    "Age 16 Matrices",
    "Age 16 Arithmetic",
    "Age 10 Reading",
    "Age 10 Maths",
    "Educational Attainment"
  )

  df %>%
    mutate(
      outcome_label = factor(outcome_label, levels = rev(outcome_order)),
      stratum_level = factor(stratum_level, levels = unique(stratum_level))
    ) %>%
    ggplot(aes(x = estimate, y = outcome_label)) +
    geom_vline(xintercept = 0, linetype = "dashed",
               colour = "grey40", linewidth = 0.5) +
    geom_errorbarh(aes(xmin = conf.low, xmax = conf.high),
                   height = 0, linewidth = 0.55,
                   colour = "black") +
    geom_point(size = 2, colour = "black", shape = 16) +
    facet_wrap(~ stratum_level, ncol = 2) +
    coord_cartesian(xlim = x_limits, clip = "on") +
    scale_x_continuous(breaks = x_breaks) +
    labs(x = "Beta (95% CI)", y = NULL, title = title) +
    my_custom_theme() +
    theme(
      legend.position  = "none",
      axis.text.y      = element_text(size = 8),
      axis.text.x      = element_text(size = 8),
      strip.text       = element_text(face = "bold", size = 9),
      strip.background = element_rect(fill = "grey92", colour = NA),
      panel.spacing    = unit(0.8, "lines"),
      plot.title       = element_text(size = 12, face = "bold")
    )
}

# ── Generate and save each stratification figure ──────────────────────────────
p_strat_socialindex <- make_strat_forest(
  strat_results$socialindex,
  title = "Cognitive outcomes stratified by parental social index"
)

p_strat_income <- make_strat_forest(
  strat_results$income,
  title = "Cognitive outcomes stratified by parental income"
)

p_strat_parent_ed <- make_strat_forest(
  strat_results$parent_ed,
  title = "Cognitive outcomes stratified by parental education"
)

# Social index has 6 strata so needs more height
ggsave("strat_forest_socialindex.png", p_strat_socialindex,
       width = 12, height = 14, dpi = 300)

ggsave("strat_forest_income.png", p_strat_income,
       width = 12, height = 10, dpi = 300)

ggsave("strat_forest_parent_ed.png", p_strat_parent_ed,
       width = 12, height = 8, dpi = 300)

## Adult education as covariate

In [ ]:
######### SENSITIVITY ANALYSIS
# Outcomes that should include adulteduc as a covariate
adult_outcomes <- c(
  "wordlist_immediate46_z", "wordlist_delayed46_z",
  "animal_naming46_z", "letter_correct46_z", "letter_speed46_z",
  "hypertension", "diabetes",
  "chol_std", "hdl_std", "hba1c_std", "igf1_std",
  "log_frtin_std", "log_trig_std", "rbc_std",
  "log_crp_std", "bmi_z", "waisthip_z",
  "bp_sys_z", "bp_dia_z", "cmvigg","vocab42_z","literacy35_z","numeracy35_z"
)
ipw_res_list <- list()   # initialise before the loop

for (y in outcomes) {
  fam <- family_map[[y]]

  if (y %in% adult_outcomes) {
    fml <- as.formula(paste(y, "~ measlesinfection + adulteduc"))
  } else {
    fml <- as.formula(paste(y, "~ measlesinfection"))
  }

  df_use <- df_ps %>% filter(!is.na(.data[[y]]), !is.na(sw_trunc))
  # ← removed the stray closing brace that was here

  if (fam == "binomial") {
    fit_w  <- glm(fml, data = df_use, family = binomial, weights = sw_trunc)
    vcov_w <- sandwich::vcovHC(fit_w, type = "HC3")
    coef_est <- coef(fit_w)
    se_rob   <- sqrt(diag(vcov_w))
    z        <- coef_est / se_rob
    pvals    <- 2 * pnorm(-abs(z))
    ci_l     <- exp(coef_est - qnorm(0.975) * se_rob)
    ci_u     <- exp(coef_est + qnorm(0.975) * se_rob)
    res_row  <- tibble(term = names(coef_est),
                       estimate = exp(coef_est), std.error = se_rob,
                       statistic = z, p.value = pvals,
                       conf.low = ci_l, conf.high = ci_u,
                       model = "IPW", outcome = y,
                       family = "binomial", n = nrow(df_use))
  } else {
    fit_w  <- lm(fml, data = df_use, weights = sw_trunc)
    vcov_w <- sandwich::vcovHC(fit_w, type = "HC3")
    coef_est <- coef(fit_w)
    se_rob   <- sqrt(diag(vcov_w))
    tstat    <- coef_est / se_rob
    pvals    <- 2 * pt(-abs(tstat), df = fit_w$df.residual)
    ci_l     <- coef_est - qt(0.975, df = fit_w$df.residual) * se_rob
    ci_u     <- coef_est + qt(0.975, df = fit_w$df.residual) * se_rob
    res_row  <- tibble(term = names(coef_est),
                       estimate = coef_est, std.error = se_rob,
                       statistic = tstat, p.value = pvals,
                       conf.low = ci_l, conf.high = ci_u,
                       model = "IPW", outcome = y,
                       family = "continuous", n = nrow(df_use))
  }

  ipw_res_list[[y]] <- res_row
}   # ← closing brace for the for loop

ipw_results_adulteduc <- bind_rows(ipw_res_list) %>%
  filter(grepl("measlesinfection", term))

In [ ]:
ipw_results_adulteduc

In [ ]:
# --- Merge crude and IPW ---
comparison_table <- crude_results %>%
  select(outcome, family,
         crude_est = estimate,
         crude_low = conf.low,
         crude_high = conf.high,
         crude_p   = p.value,
         n_crude   = n) %>%
  left_join(
    ipw_results_adulteduc %>%
      select(outcome,
             ipw_est   = estimate,
             ipw_low   = conf.low,
             ipw_high  = conf.high,
             ipw_p     = p.value,
             n_ipw     = n),
    by = "outcome"
  ) %>%
  mutate(
    effect_measure = ifelse(family == "binomial", "OR", "Beta"),
    across(c(crude_est, crude_low, crude_high,
             ipw_est,   ipw_low,   ipw_high), ~round(.x, 3)),
    crude_ci = paste0(crude_low, " - ", crude_high),
    ipw_ci   = paste0(ipw_low,   " - ", ipw_high),
    
    # ── A priori outcome groups — EDIT to match your variable names ───────────
    outcome_group = case_when(
      outcome %in% c("hypertension",
                    "diabetes")        ~ "cardiometabolic",

      outcome %in% c("cmvigg",
                     "bmi_z",
                     "bp_dia_z",
                     "bp_sys_z",
                     "chol_std",
                     "hba1c_std",
                     "hdl_std",
                     "igf1_std",
                     "log_crp_std",
                     "log_frtin_std",
                     "log_trig_std",
                     "rbc_std",
                     "waisthip_z")        ~ "biomarker",
        
      outcome %in% c("animal_naming46_z",
                     "letter_correct46_z",
                     "letter_speed46_z",
                     "maths_score10_z",
                     "maths_score16_z",
                     "matrices_score16_z",
                     "reading_score_raw10_z",
                     "reading_score_total16_z",
                     "wordlist_delayed46_z",
                     "wordlist_immediate46_z", "vocab42_z","literacy35_z","numeracy35_z") ~ "cognitive",

      TRUE ~ "other"   # catches anything not yet assigned — check this is empty
    )
  ) %>%
  # ── BH adjustment within each a priori group ─────────────────────────────────
  group_by(outcome_group) %>%
  mutate(
    crude_p_fdr = signif(p.adjust(crude_p, method = "BH"), digits = 4),
    ipw_p_fdr   = signif(p.adjust(ipw_p,   method = "BH"), digits = 4)
  ) %>%
  ungroup() %>%
  # ── Round raw p after FDR so both are consistent ──────────────────────────────
  mutate(across(c(crude_p, ipw_p), ~signif(.x, digits = 4))) %>%
  # ── Order rows by group then outcome ─────────────────────────────────────────
  arrange(outcome_group, family, outcome) %>%
  select(outcome_group, outcome, effect_measure, family,
         n_crude, crude_est, crude_ci, crude_low, crude_high, crude_p, crude_p_fdr,
         n_ipw,   ipw_est,   ipw_ci,   ipw_low,   ipw_high,   ipw_p,   ipw_p_fdr)

# ── Sanity check: anything landing in "other"? ────────────────────────────────
other_check <- comparison_table %>% filter(outcome_group == "other")
if (nrow(other_check) > 0) {
  cat("WARNING: these outcomes were not assigned to a group:\n")
  print(other_check$outcome)
} else {
  cat("All outcomes assigned to a group.\n")
}

print(comparison_table)
write.csv(comparison_table, "crude_vs_ipw_table_adulteduc.csv", row.names = FALSE)